In [3]:
!nvidia-smi

Thu May 30 16:47:36 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla K80           Off  | 00000000:05:00.0 Off |                    0 |
| N/A   69C    P0   133W / 149W |   7459MiB / 11441MiB |    100%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla K80           Off  | 00000000:06:00.0 Off |                    0 |
| N/A   

In [ ]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist,load_nifti_file,convert_file_format,file_exists, load_nifti_file_af_datatype
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class
from utils.pred_helper import remove_nrrd_files, multiply_vol_save_nifti, multiply_vol_save_nrrd

In [ ]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

project_name = 'HCFC1' #change here for different task name
task_name = 'Dataset002_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [ ]:
%env nnUNet_raw=$nnUNet_raw
%env nnUNet_preprocessed=$nnUNet_preprocessed
%env nnUNet_results=$nnUNet_results

In [ ]:
org_path = '/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/'

endswith = '.nrrd'
convert_file_format(org_path,org_path,endswith)

In [ ]:
!nnUNetv2_predict -d Dataset001_Wdr47Kusss -i /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/ -o /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/_temp/binary/ -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans

In [ ]:
#try with nifti format data. 

source_path="/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/"
binary_path="/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/_temp/binary/"
save_path="/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/_temp/roi/" 

multiply_vol_save_nifti(source_path, binary_path, save_path)

endswith = '.nrrd'
convert_file_format(save_path,save_path,endswith)

!ls ${save_path}



In [ ]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/_temp/roi/ -o /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/28-05-2024/pred/ -f  0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans

In [ ]:
remove_nrrd_files(save_path)

In [ ]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/volume/ -o /work/shared/ngmm/3Dimage/DL_test/target_seg_pred/ -f  0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans

In [ ]:
import nibabel as nib
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def empty_slice_check(data, shape): 
    print("list of empty slice")

    for slice in range(shape[2]):

        slice_data = data[:, :,slice]  # Extract the slice
        mean_value = np.mean(slice_data)   # Calculate the mean of the slice

        if mean_value == 0:
            print(slice)

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/NG2614_right_0000.nii.gz'
m_nifti_path = '/work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/test/NG2614_right.nii.gz'
img = nib.load(nifti_path)
multi_img = nib.load(m_nifti_path)

img_data = img.get_fdata()
multi_data = multi_img.get_fdata()

img_shape = img.shape
multi_shape = multi_img.shape
print('org vol: ',img_shape)
print('region vol: ',multi_shape)

print("original Image:")
empty_slice_check(img_data, img_shape )
print("region Image:")
empty_slice_check(multi_data, multi_shape )


In [ ]:
import nibabel as nib
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/scripts/Taiabur/testdata/roi24/NG4108_RCL5_0000.nii.gz'
gt_nifti_path = '/work/shared/ngmm/scripts/Taiabur/testdata/roi24/NG2613_left_01.nii.gz'
img = nib.load(nifti_path)
data = img.get_fdata()
gt_data = nib.load(gt_nifti_path).get_fdata()

# nww_data = gt_data * data
# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no_x,slice_no_y,slice_no_z):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(2,2, figsize=(16, 8))
    
    # Display the slice
    ax = axes[0,0]
    ax.imshow(data[:, :, slice_no_x])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no_x}')
    
        # Display the histogram
    ax = axes[0,1]
    slice_data = data[:, :, slice_no_x].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Img: Pixel Intensity Distribution')
    ax.grid(True)
    

    ax = axes[1,0]
    ax.imshow(data[:, slice_no_y,: ].T)
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no_y}')

    ax = axes[1,1]
    ax.imshow(data[slice_no_z, :,: ])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no_z}')

    plt.tight_layout()  # Adjust layout to not overlap
    plt.show()  # Display the plots
    
    fig, axes = plt.subplots(2,2, figsize=(16, 8))
    # gt 
    # Display the slice
    ax = axes[0,0]
    ax.imshow(gt_data[:, :, slice_no_x])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no_x}')

    # Display the histogram
    ax = axes[0,1]
    slice_data = gt_data[:, :, slice_no_x].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Img: Pixel Intensity Distribution')
    ax.grid(True)

    ax = axes[1,0]
    ax.imshow(gt_data[:, slice_no_y,: ].T)
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no_y}')

    ax = axes[1,1]
    ax.imshow(gt_data[slice_no_z, :,: ])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no_z}')



    fig.suptitle('3D Slice Viewer', fontsize=16)

    plt.tight_layout()
    plt.show()
    
# Interactive widget for slice selection
slice_slider_x = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice x'
)
# Interactive widget for slice selection
slice_slider_y = widgets.IntSlider(
    min=0, 
    max=data.shape[1] - 1, 
    step=1, 
    value=data.shape[1] // 2, 
    description='Slice y '
)
# Interactive widget for slice selection
slice_slider_z = widgets.IntSlider(
    min=0, 
    max=data.shape[0] - 1, 
    step=1, 
    value=data.shape[0] // 2, 
    description='Slice z '
)

widgets.interactive(display_slice_and_histogram, slice_no_x=slice_slider_x,slice_no_y=slice_slider_y,slice_no_z=slice_slider_z)

In [ ]:
import nibabel as nib
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path =  '/work/shared/ngmm/scripts/Taiabur/testdata/roi24/roi_24NG2614_right.nii.gz'

img = nib.load(nifti_path)
data = img.get_fdata()

# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no_x,slice_no_y,slice_no_z):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(2,2, figsize=(16, 8))
    
    mid_slice = data.shape[2] // 2

    print(mid_slice)
    
    # Display the slice
    ax = axes[0,0]
    ax.imshow(data[:, :, slice_no_x])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no_x}')
    
        # Display the histogram
    ax = axes[0,1]
    ax.grid(True)
    

    ax = axes[1,0]
    ax.imshow(data[:, slice_no_y,: ].T)
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no_y}')

    ax = axes[1,1]
    ax.imshow(data[slice_no_z, :,: ])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no_z}')

    fig.suptitle('3D Slice Viewer', fontsize=16)

    plt.tight_layout()
    plt.show()

# Interactive widget for slice selection
slice_slider_x = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice x'
)
# Interactive widget for slice selection
slice_slider_y = widgets.IntSlider(
    min=0, 
    max=data.shape[1] - 1, 
    step=1, 
    value=data.shape[1] // 2, 
    description='Slice y '
)
# Interactive widget for slice selection
slice_slider_z = widgets.IntSlider(
    min=0, 
    max=data.shape[0] - 1, 
    step=1, 
    value=data.shape[0] // 2, 
    description='Slice z '
)


widgets.interactive(display_slice_and_histogram, slice_no_x=slice_slider_x,slice_no_y=slice_slider_y,slice_no_z=slice_slider_z)